# 02 — Train CNN + ViT and compare

Use the CLI for full runs; here is the programmatic path.

In [ ]:
from waferlens.config import load_config
from waferlens.data.mixedwm38 import load_npz
from waferlens.data.dataset import make_splits
from waferlens.models.factory import build_model, count_params
from waferlens.train.loop import train_model

cfg = load_config('configs/config.yaml')
d = load_npz('data/synthetic_sample.npz')
splits = make_splits(d.X, d.Y, d.classes, 0.15, 0.15, 42, True, True, True)

results = {}
for name in ['cnn', 'vit']:
    net = build_model(name, 8, cfg.model, 52)
    results[name] = train_model(net, splits, model_name=name, epochs=5, batch_size=128,
        lr=3e-4, weight_decay=1e-4, device_str='auto', num_workers=0,
        early_stop_patience=8, threshold=0.5,
        checkpoint_dir='artifacts/checkpoints', n_params=count_params(net))
for name, r in results.items():
    print(name, r.n_params, 'params ->', {k: round(v,3) for k,v in r.test_metrics.items() if isinstance(v,float)})
